# Retrieval-Augmented Generation (RAG)

In [1]:
import chromadb
import dotenv
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [15]:
# We populated the RAG with the data from the data/calories.csv file in
# the rag_setup.ipynb notebook
chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db =chroma_client.get_collection(name="nutrition_db")
nutrition_qna_db = chroma_client.get_collection(name="nutrition_qna")

In [3]:
results = nutrition_db.query(query_texts=["banana"], n_results=2)
for i, doc in enumerate(results["documents"][0]):
    print(sorted(results["metadatas"][0][i].items()))
    print(doc)
    print("\n")

/home/vscode/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 33.9MiB/s]


[('calories_per_100g', 89.0), ('food_category', 'fruits'), ('food_item', 'banana'), ('keywords', 'banana_fruits'), ('kj_per_100g', 374.0), ('serving_info', '100g')]
Food: Banana
        Category: Fruits
        Nutritional Information:
        - Calories: 89 per 100g
        - Energy: 374 kJ per 100g
        - Serving size reference: 100g

        This is a fruits food item that provides 89 calories per 100 grams.


[('calories_per_100g', 50.0), ('food_category', '(fruit)juices'), ('food_item', 'banana juice'), ('keywords', 'banana_juice_(fruit)juices'), ('kj_per_100g', 210.0), ('serving_info', '100ml')]
Food: Banana Juice
        Category: (Fruit)Juices
        Nutritional Information:
        - Calories: 50 per 100g
        - Energy: 210 kJ per 100g
        - Serving size reference: 100ml

        This is a (fruit)juices food item that provides 50 calories per 100 grams.




In [20]:
# Test the setup with sample queries
chroma_client = chromadb.PersistentClient("../chroma")
nutrition_qna = chroma_client.get_collection(name="nutrition_qna")

# Test query 1: Search for malnutrition symptoms
print("=== Q/A Query ===")
qna_results = nutrition_qna_db.query(query_texts=["Malnutrition"], n_results=2)
for index, doc in enumerate(results["documents"] [0]):
    print(f"Question {index +1}: ")
    print(f"Answer: {results['documents'][0]}")
    print("\n")

=== Q/A Query ===
Question 1: 
Answer: ['Question: What are some possible physical changes that a pregnant woman could experience in the middle months of gestation?\n        Answer: During weeks 13-27, you may see an increase in weight and feel more hunger. Backaches might occur frequently, along with leg cramps and heartburn.\n\n        This Q&A pair provides information about nutrition and health topics.', 'Question: What health issues can make a pregnancy more challenging?\n        Answer: There are several medical conditions that can potentially increase the risk associated with Pregnancy. These include Anemia during this stage, Hypertensive Disorders related to gestation, Diabetes Mellitus coexisting with Pregnancy, Obesity during pregnancy, as well as Adolescent or Teenage pregnancies.\n\n        This Q&A pair provides information about nutrition and health topics.', "Question: What are some hormonal changes that take place in a mother's body during pregnancy?\n        Answer: Du

In [7]:
@function_tool
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [8]:
calorie_lookup_tool('bananas')

TypeError: 'FunctionTool' object is not callable

In [27]:
@function_tool
def qna_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up Nutrition Q&A database with questions and answers about nutrition and health information for a specific question.

    Args:
        query: The question to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the Answer about the asked question.
    """

    results = nutrition_qna_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No qna question found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        answer = metadata["answer"].title

        formatted_results.append(
            f"{answer}"
        )

    return f"The answer to is the question {query} are:\n" + "\n".join(formatted_results)

In [28]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information and nutrition Q&A response about Q&A question.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool.
    If you need to lookup question about Nutrition, use the qna_lookup_tool

    """,
    tools= [calorie_lookup_tool, qna_lookup_tool]
)

In [24]:
with trace("Nutrition Assistant with RAG"):
    result = await Runner.run(
        calorie_agent,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )
    print(result.final_output)

- Per 100 g: Banana 89 kcal; Apple 52 kcal.

- For typical sizes (approximate):
  - Medium banana (~118 g): ~105 kcal
  - Medium apple (~182 g): ~95 kcal
  - Total ≈ 200 kcal

If you have specific weights, I can calculate precisely.


In [31]:
with trace("Nutrition Q&A Assistant with RAG"):
    result = await Runner.run(calorie_agent, "What are the possible consequences on a child's growth due to lack of proper food")

    print(result.final_output)

Lack of proper food in children can lead to several serious outcomes:

- Growth issues: stunting (short stature for age), wasting (low weight for height), underweight.
- Developmental and cognitive impacts: delayed mental and motor development, learning difficulties, reduced school performance.
- Immune and health problems: weakened immunity, higher infection risk, slower recovery.
- Micronutrient deficiencies: anemia (iron), iodine deficiency (c cognitive/thyroid effects), zinc (growth and immune issues), vitamin A (vision and immunity), B vitamins (energy/metabolism).
- Hormonal and puberty effects: delayed or disrupted puberty.
- Long-term health risk: increased risk of chronic diseases later in life.

If this is ongoing, consult a healthcare provider or nutritionist for assessment and a tailored plan.
